# SageMaker Tabular Classification

In [ ]:
%cd ~/SageMaker/
!python3 -m pip install --upgrade "sagemaker<3" pandas boto3

In [ ]:
# URL만 변경
!curl -L "<DOWNLOAD_URL>" -o dataset.zip

In [ ]:
!unzip -o dataset.zip -d dataset
!find dataset -type f

In [ ]:
# CSV 경로만 변경
import pandas as pd

df = pd.read_csv("dataset/Iris.csv")
# 헤더가 없으면:
# df = pd.read_csv("dataset/Iris.csv", header=None, names=["feature_1", "feature_2", "target"])

df.info()
display(df.head())
display(df.isnull().sum())

In [ ]:
import sagemaker
from sagemaker import get_execution_role

session = sagemaker.Session()
bucket = session.default_bucket()
role = get_execution_role()

In [ ]:
from sagemaker.s3 import S3Uploader

inputs = S3Uploader.upload(
    "dataset/Iris.csv",  # CSV 경로 변경
    f"s3://{bucket}/sagemaker/dataset",
)
print(inputs)

In [ ]:
from sagemaker.xgboost import XGBoost

estimator = XGBoost(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="3.0-5",
    output_path=f"s3://{bucket}/sagemaker/dataset/output/",
)

estimator.fit({"train": inputs})

In [ ]:
# Endpoint 이름만 변경
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="<ENDPOINT_NAME>",
)

In [ ]:
import boto3
import json

# 학습에 사용한 feature 이름과 실제 값 입력
body = {
    "instances": [
        {"SepalLengthCm": 5.1, "SepalWidthCm": 3.5, "PetalLengthCm": 1.4, "PetalWidthCm": 0.2}
    ]
}

response = boto3.client("sagemaker-runtime").invoke_endpoint(
    EndpointName="<ENDPOINT_NAME>",
    ContentType="application/json",
    Body=json.dumps(body),
)
print(json.load(response["Body"]))

In [ ]:
# 사용 후 실행하여 Endpoint 비용 중지
# predictor.delete_endpoint(delete_endpoint_config=True)
# predictor.delete_model()